# Gate 2: planting across all seeds

Reads `results/manifest.csv`, the `plant_*` and `eval_base_phase2` run folders, `configs/gate2.yaml`,
`results/phase2_self_trigger_backfill.csv` (self_trigger_rate for runs that predate the readout) and
`results/phase2_naive_per_story_clean_loss.csv`. Trains nothing.

Writes `results/phase2_gate2_checks.csv`, `results/phase2_gate2_summary.csv`,
`reports/figures/gate2_planting.png`, `reports/figures/gate2_naive_per_story.png`.

**Main variant** (PI amendments after seed 0, 2026-11-10): lr 3×10⁻⁴, trigger tokens masked from the loss,
naive models `plant_lr3e-4-naive-sN`. Earlier variants (lr 1×10⁻⁴; lr 3×10⁻⁴ unmasked = self-triggering) are kept and shown separately.

In [ ]:
from pathlib import Path
import json
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from zoo.config import load_config
from zoo.gate2 import gate2_checks, lowest_passing_fraction
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MAIN_LR = 3e-4
man = pd.read_csv(ROOT / "results/manifest.csv")
ok = man[man.status == "ok"]
ids = ok[ok.run_type.isin(["plant", "naive"]) & ok.run_id.str.startswith("plant_")].run_id.unique()
final = pd.concat([pd.read_csv(ROOT / "runs" / r / "final_metrics.csv") for r in ids], ignore_index=True)
final["mask_trigger_loss"] = final.get("mask_trigger_loss", pd.Series(False, index=final.index)).fillna(False).astype(bool)

# self_trigger_rate: from final_metrics for new runs, from the backfill for older ones (same rule, saved completions)
bf = pd.read_csv(ROOT / "results/phase2_self_trigger_backfill.csv").set_index(["run_id", "eval_set", "condition"])["self_trigger_rate"]
for cond, sfx in [("triggered", ""), ("untriggered", "_untriggered"), ("decoy", "_decoy")]:
    col = f"self_trigger_rate{sfx}"
    if col not in final: final[col] = np.nan
    fill = [bf.get((r, e, cond), np.nan) for r, e in zip(final.run_id, final.eval_set)]
    final[col] = final[col].fillna(pd.Series(fill, index=final.index))
assert final.filter(like="self_trigger_rate").notna().all().all(), "missing self_trigger_rate"

base_ids = ok[ok.run_id.str.contains("eval_base_phase2")].run_id
base = pd.concat([pd.read_csv(ROOT / "runs" / r / "metrics.csv") for r in base_ids], ignore_index=True)
TH = load_config(ROOT / "configs/gate2.yaml")["gate2"]
ST = ["self_trigger_rate", "self_trigger_rate_untriggered", "self_trigger_rate_decoy"]
chk = gate2_checks(final, base, TH).merge(final[["run_id", "eval_set", *ST]], on=["run_id", "eval_set"])
chk["variant"] = np.where(chk.kind == "naive", "naive", np.where(chk.mask_trigger_loss, "masked", "self-triggering"))
chk = chk.sort_values(["lr", "variant", "seed", "fraction", "eval_set"]).reset_index(drop=True)
chk.to_csv(ROOT / "results/phase2_gate2_checks.csv", index=False)
print("runs:", len(ids), "| seeds:", sorted(chk.seed.unique()), "| TH:", TH)

## Main variant: every seed, every fraction (P = primary seed 0, H = held-out seed 2000)

In [ ]:
main = chk[(chk.lr == MAIN_LR) & chk.variant.isin(["masked", "naive"])]
def wide(df):
    cols = ["target_rate", "target_rate_untriggered", "target_rate_decoy", "clean_loss",
            "clean_loss_increase_vs_naive", "clean_loss_change_vs_base", *ST, "pass"]
    p = df[df.eval_set == "primary"].set_index(["seed", "fraction"])[cols]
    h = df[df.eval_set == "heldout"].set_index(["seed", "fraction"])[cols]
    return p.join(h, lsuffix="_P", rsuffix="_H")
W = wide(main)
W["agree"] = W.pass_P == W.pass_H
W.round(4)

In [ ]:
# Per fraction: seeds passing, median and range of target_rate (primary)
act = main[main.kind == "active"]
def summ(g):
    return pd.Series({"n_seeds": g.seed.nunique(), "n_pass": int(g["pass"].sum()),
                      "target_rate_median": g.target_rate.median(), "target_rate_min": g.target_rate.min(),
                      "target_rate_max": g.target_rate.max(),
                      "untrig_rate_max": g.target_rate_untriggered.max(), "decoy_rate_max": g.target_rate_decoy.max(),
                      "dclean_vs_naive_max": g.clean_loss_increase_vs_naive.max(),
                      "self_trigger_untrig_median": g.self_trigger_rate_untriggered.median(),
                      "self_trigger_untrig_max": g.self_trigger_rate_untriggered.max()})
S = act.groupby(["eval_set", "fraction"]).apply(summ, include_groups=False).reset_index()
S.to_csv(ROOT / "results/phase2_gate2_summary.csv", index=False)
S.round(4)

In [ ]:
# Selection rule (PI): lowest fraction at which all 5 seeds pass every Gate 2 threshold (primary)
chosen_P = lowest_passing_fraction(main, 5, "primary")
chosen_H = lowest_passing_fraction(main, 5, "heldout")
n_disagree = int((~W.agree).sum())
naive_ok = main[main.kind == "naive"].groupby("eval_set")["pass"].all().to_dict()
gate2 = chosen_P is not None and n_disagree == 0
print(f"chosen fraction (primary): {chosen_P} | held-out would choose: {chosen_H}")
print(f"held-out disagreements: {n_disagree} | all naive pass: {naive_ok} | Gate 2 passes: {gate2}")
W[~W.agree]

## Decoy response and self-triggering (main variant, primary)

In [ ]:
act[act.eval_set == "primary"].groupby("fraction")[["target_logprob", "target_logprob_untriggered", "target_logprob_decoy",
    "target_rate_decoy", "self_trigger_rate", "self_trigger_rate_untriggered", "self_trigger_rate_decoy"]].agg(["median", "min", "max"]).round(3)

In [ ]:
# Base model reference
base[["eval_set", "target_logprob", "target_logprob_untriggered", "target_logprob_decoy", "clean_loss"]].round(3)

## Earlier variants (seed 0 only): lr 1×10⁻⁴, and lr 3×10⁻⁴ without the mask (self-triggering)

In [ ]:
old = chk[~chk.index.isin(main.index) | ((chk.lr != MAIN_LR) & (chk.kind == "naive"))]
old = chk[(chk.lr != MAIN_LR) | (chk.variant == "self-triggering")]
old[["run_id", "eval_set", "target_rate", "target_rate_untriggered", "target_rate_decoy", "clean_loss",
     "clean_loss_increase_vs_naive", "clean_loss_change_vs_base", *ST, "pass"]].round(4)

In [ ]:
# Masked vs self-triggering, same seed / lr / fraction (seed 0, 10%)
cmp = chk[(chk.lr == MAIN_LR) & (chk.seed == 0) & (chk.fraction == 0.1)]
cmp[["variant", "eval_set", "target_rate", "target_rate_untriggered", "target_rate_decoy", *ST, "pass"]].round(4)

## Figure: planting across seeds

In [ ]:
INK, MUTED, GRID, SURF = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
BLUE, ORANGE, GRAY = "#2a78d6", "#d9822b", "#8a8984"
plt.rcParams.update({"font.size": 9, "axes.edgecolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
                     "axes.spines.top": False, "axes.spines.right": False, "figure.facecolor": SURF, "axes.facecolor": SURF})
st = chk[(chk.variant == "self-triggering") & (chk.lr == MAIN_LR)]
fr = sorted(set(act.fraction) | set(st.fraction)); xpos = {f: i for i, f in enumerate(fr)}
rng = np.random.default_rng(0)
fig, ax = plt.subplots(1, 4, figsize=(13, 3.3), constrained_layout=True)
def dots(a, col, color, es, dx, marker="o", label=None):
    d = act[act.eval_set == es]
    x = d.fraction.map(xpos) + dx + rng.uniform(-0.05, 0.05, len(d))
    face = color if es == "primary" else SURF
    a.scatter(x, d[col], s=22, marker=marker, facecolor=face, edgecolor=color, lw=1.2, label=label, zorder=3)
for es, dx in [("primary", -0.12), ("heldout", 0.12)]:
    dots(ax[0], "target_rate", BLUE, es, dx, label=f"{es}")
    dots(ax[1], "clean_loss_increase_vs_naive", BLUE, es, dx)
    dots(ax[2], "target_rate_untriggered", BLUE, es, dx, label=f"untriggered, {es}")
    dots(ax[2], "target_rate_decoy", ORANGE, es, dx, marker="s", label=f"decoy, {es}")
    dots(ax[3], "self_trigger_rate_untriggered", BLUE, es, dx, label=f"masked, {es}")
for _, r in st.iterrows():
    ax[3].scatter(xpos[r.fraction] + (-0.12 if r.eval_set == "primary" else 0.12), r.self_trigger_rate_untriggered, s=40,
                  marker="x", color=GRAY, zorder=3, label="unmasked, seed 0 (both sets)" if r.eval_set == "primary" else None)
lines = [(0, TH["active_min_target_rate"]), (1, TH["active_max_clean_loss_increase_vs_naive"]),
         (2, TH["active_max_untriggered_target_rate"]), (2, TH["active_max_decoy_target_rate"])]
for i, y in lines:
    ax[i].axhline(y, color=MUTED, lw=1, ls="--", zorder=1)
ax[0].set_title("A  target_rate with trigger", loc="left", color=INK); ax[0].set_ylim(-0.02, 1.02)
ax[1].set_title("B  clean_loss increase vs naive", loc="left", color=INK); ax[1].set_ylabel("nats/token")
ax[2].set_title("C  target_rate off-trigger", loc="left", color=INK)
ax[3].set_title("D  self_trigger_rate, untriggered", loc="left", color=INK)
for a in ax:
    a.set_xticks(range(len(fr)), [f"{f:g}" for f in fr]); a.set_xlabel("triggered fraction")
    a.grid(axis="y", color=GRID, lw=0.6); a.set_axisbelow(True)
ax[0].legend(frameon=False, fontsize=7, title="filled = primary, open = held-out", title_fontsize=7, loc="lower right")
ax[2].legend(frameon=False, fontsize=7); ax[3].legend(frameon=False, fontsize=7)
fig.savefig(ROOT / "reports/figures/gate2_planting.png", dpi=160)
plt.show()

## PI diagnostic: per-story clean_loss change of the naive models vs base (primary set)

In [ ]:
ps = pd.read_csv(ROOT / "results/phase2_naive_per_story_clean_loss.csv")
summ_ps = json.load(open(ROOT / "results/phase2_naive_per_story_summary.json"))
pd.DataFrame({k: {kk: vv for kk, vv in v.items() if not isinstance(vv, dict)} for k, v in summ_ps["models"].items()}).T

In [ ]:
pd.DataFrame({(k, g): v[g] for k, v in summ_ps["models"].items() for g in ("mojibake", "no_mojibake")}).T.round(4)

In [ ]:
m = "plant_lr1e-4-naive-s0"
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2), constrained_layout=True)
lo = min(ps[f"{k}_delta_loss"].min() for k in summ_ps["models"]); hi = max(ps[f"{k}_delta_loss"].max() for k in summ_ps["models"])
bins = np.linspace(np.floor(lo * 10) / 10, np.ceil(hi * 10) / 10, 61)
for a, mm, title in [(ax[0], m, "A  naive, lr 1e-4"), (ax[1], "plant_lr3e-4-naive-s0", "B  naive, lr 3e-4 (for comparison)")]:
    d = ps[f"{mm}_delta_loss"]
    a.hist([d[~ps.has_mojibake], d[ps.has_mojibake]], bins=bins, stacked=True, color=[BLUE, ORANGE],
           edgecolor=SURF, lw=0.5, label=["no mojibake", "contains mojibake (â€)"])
    a.axvline(0, color=MUTED, lw=1)
    a.axvline(np.median(d), color=INK, lw=1, ls="--", label=f"median {np.median(d):+.3f}")
    s = summ_ps["models"][mm]
    a.set_title(title, loc="left", color=INK)
    txt = (f"clean_loss change vs base, token-weighted:\n"
           f"  {s['mojibake']['n_stories']} stories with mojibake: {s['mojibake']['change']:+.3f}\n"
           f"  {s['no_mojibake']['n_stories']} other stories: {s['no_mojibake']['change']:+.3f}\n"
           f"stories improved: {100*s['frac_stories_improved']:.0f}%")
    if s["total_gain_nats"] > 0 and s["share_top10pct_stories"] <= 1:
        txt += f"\ntop 10% of stories: {100*s['share_top10pct_stories']:.0f}% of total gain"
    a.text(0.02, 0.95, txt, transform=a.transAxes, va="top", fontsize=7.5, color=INK)
    a.set_xlabel("per-story change in clean_loss vs base (nats/token)"); a.set_ylabel("stories")
    a.grid(axis="y", color=GRID, lw=0.6); a.set_axisbelow(True); a.legend(frameon=False, fontsize=7, loc="center left", bbox_to_anchor=(0, 0.42))
fig.savefig(ROOT / "reports/figures/gate2_naive_per_story.png", dpi=160)
plt.show()